<a href="https://colab.research.google.com/github/jottanovaes/colab-notebooks/blob/trabalho-final-programacao-linear/trabalho_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problema 6 - Escala de funcionários
Uma empresa possui uma equipe que trabalha sete dias por semana em turnos de trabalho de
mesmo tempo.

A equipe possui *n* funcionários. Cada funcionário deve ter 5 folgas durante o
mês e para garantir que o trabalho seja realizado de forma competente todos os dias, a equipe
não pode operar com menos que *l* funcionários.

Por razões contratuais nenhum funcionário pode
trabalhar por 6 dias seguidos sem folgas e ao menos uma das folgas deve ocorrer em um domingo
ou em um feriado. Os funcionários são consultados e indicam datas preferenciais para folgas.
Construa um modelo de otimização que determine a escala dos funcionários para o mês de junho
de 2026 que:

1.   Maximize o número total de folgas nos dias requisitados pelos funcionários.
2.   Distribua as folgas nos dias requisitados de maneira mais equilibrada possível entre os funcionários.

# 1. Maximize o número total de folgas nos dias requisitados pelos funcionários.

## modelo algebrico
- *n* funcionarios
- *m* dia do mês

## variaveis de decisao
$$
x_{ij} =
\begin{cases}
1, & \text{se o funcionário folga no dia} \\
0, & \text{caso contrário}
\end{cases}
$$

## parametros
$$ l = \text{minimo de funcionarios em cada turno} $$
$$ D = \text{domingos e feriados} $$
$$ P_{ij} = \text{preferencia do funcionario } i \text{ para folgar no dia } j $$

## funcao objetivo
$$
 max  \sum_{i=1}^{n}\sum_{j=1}^{m} P_{ij} \cdot x_{ij}
$$

## retricoes
$$
\sum_{j=1}^{m} x_{ij} = 5, \quad \forall i \in \{1, \ldots, n\} \\
\sum_{i=1}^{n} (1 - x_{ij}) \ge l, \quad \forall j \in \{1, \ldots, m\} \\
\sum_{j \in D} x_{ij} \ge 1, \quad \forall i \in \{1, \ldots, n\} \\
\sum_{k=j}^{j+5} x_{ik} \ge 1, \quad \forall i \in \{1,\ldots,n\},\ \forall j \in \{1, \ldots, m-5\} \\
x_{ij} \in \{0, 1\}, \quad \forall i, \forall j
$$

In [43]:
!pip install gurobipy

import pandas as pd
import requests
import gurobipy as gp
import time
import numpy as np
from google.colab import files

In [44]:
# calendario junho/2026
D = {3, 6, 13, 20, 27} # indice dos dias que sao domingos o feriados
m = 30

# dados de entrada
instancias = [
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_20_15.txt",
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_25_19.txt",
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_30_23.txt",
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_35_27.txt",
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_40_31.txt",
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_45_35.txt",
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_50_39.txt",
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_60_47.txt",
]

In [45]:
def ler_instancia(url):
    texto = requests.get(url).text
    linhas = [l.strip() for l in texto.strip().splitlines() if l.strip()]

    n = int(linhas[0]) # numero de funcionarios
    l = int(linhas[1]) # numero minimo de funcionarios em cada turno

    P = [] # preferencias
    for i in range(2, 2 + n):
        dias = [int(d) - 1 for d in linhas[i].split()]
        P.append(dias)

    P_matrix = [
        [1 if j in P[i] else 0 for j in range(m)]
        for i in range(n)
    ] # cria matriz binária n x m, 1 se o funcionario i quer folgar no dia j


    return n, l, P, P_matrix

In [46]:
def otimizar(n, l, m, P, P_matrix, D):
  modelo = gp.Model()

  x = modelo.addVars(
      n,
      m,
      vtype = gp.GRB.BINARY,
      name = "x"
  )

  modelo.setObjective(
      gp.quicksum(P_matrix[i][j] * x[i, j] for i in range(n) for j in range(m)),
      gp.GRB.MAXIMIZE
  )

  # r1: cada funcionario tem exatamente 5 folgas no mes
  r1 = modelo.addConstrs(
      gp.quicksum(x[i, j] for j in range(m)) == 5
      for i in range(n)
  )

  # r2: minimo de l funcionarios trabalhando por dia
  r2 = modelo.addConstrs(
      gp.quicksum(1 - x[i, j] for i in range(n)) >= l
      for j in range(m)
  )

  # r3: pelo menos uma folga em domingo ou feriado
  r3 = modelo.addConstrs(
      gp.quicksum(x[i, j] for j in D) >= 1
      for i in range(n)
  )

  # r4: nenhum funcionario trabalha 6 dias seguidos sem folga
  # para uma janela de 7 dias, pelo menos 1 folga
  r4 = modelo.addConstrs(
      gp.quicksum(x[i, k] for k in range(j, j + 7)) >= 1
      for i in range(n)
      for j in range(m - 6)
  )

  t0 = time.time()

  modelo.setParam("OutputFlag", 0)
  modelo.optimize()
  tempo = round(time.time() - t0, 3)

  return modelo, x, tempo

In [47]:
rows_func = []      # um registro por funcionário por instância
rows_inst = []      # um registro por instância

resultados = []

for url in instancias:
    nome = url.split("/")[-1].replace(".txt", "")

    n, l, P, P_matrix = ler_instancia(url)

    modelo, x, tempo = otimizar(n, l, m, P, P_matrix, D)

    if modelo.status == gp.GRB.OPTIMAL:
        status     = "otimo"
        obj_valor  = int(modelo.objVal)
        pcts       = []

        for i in range(n):
            folgas     = [j + 1 for j in range(m) if round(x[i, j].x) == 1]
            preferidas = [j + 1 for j in range(m) if round(x[i, j].x) == 1 and j in P[i]]
            total_pref = len(P[i])
            pct        = round(len(preferidas) / total_pref * 100, 1) if total_pref else 0.0
            tem_dom    = int(any((j + 1) in folgas for j in D))
            pcts.append(pct)

            rows_func.append({
                "instancia"            : nome,
                "n"                    : n,
                "l"                    : l,
                "funcionario"          : i + 1,
                "folgas_atribuidas"    : ",".join(map(str, folgas)),
                "preferencias_atendidas": ",".join(map(str, preferidas)),
                "total_preferencias"   : total_pref,
                "pct_atendida"         : pct,
                "tem_folga_domingo"    : tem_dom,
            })

        rows_inst.append({
            "instancia"        : nome,
            "modelo"           : "A",
            "n"                : n,
            "l"                : l,
            "status"           : status,
            "obj_valor"        : obj_valor,
            "media_pct_atendida": round(np.mean(pcts), 2),
            "desvio_pct"       : round(np.std(pcts), 2),
            "min_pct"          : min(pcts),
            "max_pct"          : max(pcts),
            "tempo_execucao_s" : tempo,
        })

        print(f"  status: {status} | obj: {obj_valor} | média %: {rows_inst[-1]['media_pct_atendida']} | desvio: {rows_inst[-1]['desvio_pct']}")

    else:
        status = "infactivel"
        print(f"  status: {status}")

        rows_inst.append({
            "instancia"        : nome,
            "modelo"           : "A",
            "n"                : n,
            "l"                : l,
            "status"           : status,
            "obj_valor"        : None,
            "media_pct_atendida": None,
            "desvio_pct"       : None,
            "min_pct"          : None,
            "max_pct"          : None,
            "tempo_execucao_s" : tempo,
        })

df_func = pd.DataFrame(rows_func)
df_inst = pd.DataFrame(rows_inst)

df_func.to_csv("resultados_funcionarios.csv", index=False, header=False)
df_inst.to_csv("resultados_instancias.csv",   index=False, header=False)

print("\n✓ resultados_funcionarios.csv")
files.download("resultados_funcionarios.csv")

print("✓ resultados_instancias.csv")
files.download("resultados_instancias.csv")

display(df_inst)

  status: otimo | obj: 57 | média %: 71.25 | desvio: 16.35
  status: otimo | obj: 75 | média %: 75.0 | desvio: 15.81
  status: otimo | obj: 90 | média %: 75.0 | desvio: 15.81
  status: otimo | obj: 103 | média %: 73.57 | desvio: 14.57
  status: otimo | obj: 120 | média %: 75.0 | desvio: 16.77
  status: otimo | obj: 131 | média %: 72.78 | desvio: 13.77
  status: otimo | obj: 142 | média %: 71.0 | desvio: 15.3
  status: otimo | obj: 180 | média %: 75.0 | desvio: 15.14

✓ resultados_funcionarios.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ resultados_instancias.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,instancia,modelo,n,l,status,obj_valor,media_pct_atendida,desvio_pct,min_pct,max_pct,tempo_execucao_s
0,inst_20_15,A,20,15,otimo,57,71.25,16.35,50.0,100.0,0.025
1,inst_25_19,A,25,19,otimo,75,75.00,15.81,50.0,100.0,0.029
2,inst_30_23,A,30,23,otimo,90,75.00,15.81,50.0,100.0,0.028
3,inst_35_27,A,35,27,otimo,103,73.57,14.57,50.0,100.0,0.044
4,inst_40_31,A,40,31,otimo,120,75.00,16.77,50.0,100.0,0.036
5,inst_45_35,A,45,35,otimo,131,72.78,13.77,50.0,100.0,0.078
6,inst_50_39,A,50,39,otimo,142,71.00,15.30,50.0,100.0,0.137
7,inst_60_47,A,60,47,otimo,180,75.00,15.14,50.0,100.0,0.121


# 2. Distribua as folgas nos dias requisitados de maneira mais equilibrada possível entre os funcionários.